In [ ]:
from metabolic_gene_graph_kernel import MetabolicGeneGraphKernel
from essential.fba import load_ecoli_rich_medium_model
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotnine as gg
from fba_utils import to_long_no_diagonal
import scipy.stats as stats
from sklearn.decomposition import PCA


SHARED_THEME = gg.theme(
    axis_text=gg.element_text(size=6),
    axis_title=gg.element_text(size=7),
    figure_size=(3, 2),
    title=gg.element_text(size=7),
    legend_text=gg.element_text(size=6),
)


def compare_distances(transcript_distance_name):
    joint_dists_ = joint_dists.copy()

    corr_ = stats.spearmanr(
        joint_dists_["distance_metabolic"], joint_dists_[transcript_distance_name]
    )[0]

    fig = (
        gg.ggplot(joint_dists_, gg.aes(x="distance_metabolic", y=transcript_distance_name))
        + gg.geom_point(size=0.5)
        # + gg.geom_boxplot()
        + gg.theme_minimal()
        + gg.labs(x="Metabolic distance", title=f"Spearman rho = {corr_:.2f}")
        + SHARED_THEME
        + gg.theme(figure_size=(3, 2))
    )
    display(fig)

In [ ]:
def compute_centered_kernel(K):
    H = np.eye(K.shape[0]) - np.ones((K.shape[0], K.shape[0])) / K.shape[0]
    return H @ K @ H


def compute_cka(K, L):
    K_centered = compute_centered_kernel(K)
    L_centered = compute_centered_kernel(L)

    K_norm = np.linalg.norm(K_centered, "fro")
    L_norm = np.linalg.norm(L_centered, "fro")
    return np.trace(K_centered @ L_centered) / (K_norm * L_norm)


def compute_cosine_similarity(K, L):
    return np.trace(K @ L) / (np.linalg.norm(K, "fro") * np.linalg.norm(L, "fro"))

In [ ]:
model = load_ecoli_rich_medium_model()

In [ ]:
# msd = MetabolicGeneGraphKernel(model, currency, beta=1)

In [ ]:
from fba_utils import compute_pairwise
import scanpy as sc
from tqdm import tqdm

adata = sc.read_h5ad(
    "/workspace/data/251117_genomescale_CRISPRi/sample_mix_umi200_hvg500_pc25_neighbors10_mindist0.55.scvi.h5ad"
)
adata_case = sc.read_h5ad("/workspace/data/251117_genomescale_CRISPRi/adata_case.annotated.h5ad")

transcript_df = []
transcript_case_df = []
z_transcript_df = []
z_transcript_case_df = []
gene_names = []
gene_case_names = []


for gene in tqdm(adata.obs["gene"].unique()):
    adata_gene = adata[adata.obs["gene"] == gene]
    X_gene = adata_gene.layers["cp10k"].toarray()
    if X_gene.shape[0] > 0:
        gene_names.append(gene)
        transcript_df.append(X_gene.mean(axis=0))
        z_transcript_df.append(adata_gene.obsm["X_scVI"].mean(axis=0))

    adata_gene_case = adata_case[adata_case.obs["gene"] == gene]
    X_gene_case = adata_gene_case.layers["cp10k"].toarray()
    if X_gene_case.shape[0] > 0:
        gene_case_names.append(gene)
        transcript_case_df.append(X_gene_case.mean(axis=0))
        z_transcript_case_df.append(adata_gene_case.obsm["X_scVI"].mean(axis=0))
transcript_df = pd.DataFrame(transcript_df, index=gene_names)
transcript_case_df = pd.DataFrame(transcript_case_df, index=gene_case_names)
z_transcript_df = pd.DataFrame(z_transcript_df, index=gene_names)
z_transcript_case_df = pd.DataFrame(z_transcript_case_df, index=gene_case_names)

transcript_df_pca = PCA(n_components=50).fit_transform(transcript_df)
transcript_df_pca_ = pd.DataFrame(transcript_df_pca, index=gene_names)
transcript_pairwise_pc = compute_pairwise(transcript_df_pca_, metric="euclidean")
distance_pc = to_long_no_diagonal(transcript_pairwise_pc).rename(
    columns={"distance": "distance_pc"}
)

transcript_df_pca_case = PCA(n_components=50).fit_transform(transcript_case_df)
transcript_df_pca_case_ = pd.DataFrame(transcript_df_pca_case, index=gene_case_names)
transcript_pairwise_pc_case = compute_pairwise(transcript_df_pca_case_, metric="euclidean")
distance_pc_case = to_long_no_diagonal(transcript_pairwise_pc_case).rename(
    columns={"distance": "distance_pc_case"}
)

transcript_pairwise_z_case = compute_pairwise(z_transcript_case_df, metric="euclidean")
distance_z_case = to_long_no_diagonal(transcript_pairwise_z_case).rename(
    columns={"distance": "distance_z_case"}
)

transcript_distances = distance_pc.merge(distance_pc_case, on=["gene1", "gene2"]).merge(
    distance_z_case, on=["gene1", "gene2"]
)

K_transcript = transcript_pairwise_pc @ transcript_pairwise_pc.T

In [ ]:
# [DO NOT DELETE]
# # compute metabolite degree
# self = msd
# valid_metabolites = [m for m in self.model.metabolites if m.id in self.metabolite_to_idx]

# # Build array of 1/deg(m)
# num_metabolites = len(self.metabolite_to_idx)
# degrees = np.zeros(num_metabolites)
# for m in valid_metabolites:
#     idx = self.metabolite_to_idx[m.id]
#     deg = len(m.reactions)
#     degrees[idx] = deg

# plot_df = pd.DataFrame({"degree": degrees})
# fig = (
#     gg.ggplot(plot_df, gg.aes(x="degree"))
#     + gg.geom_histogram(bins=50)
#     + gg.theme_minimal()
#     + SHARED_THEME
#     # + gg.scale_x_log10()
#     + gg.labs(
#         x="metabolite degree",
#         y="# of metabolites",
#         title="degree distribution of non-currency metabolites\n in the metabolic network",
#     )
# )
# fig.save("metabolite_degree_distribution.png")

### $\beta$ parameter selection

In [ ]:
plot_df = []

for beta in [0.1, 1, 10, 25, 50, 100]:
    msd = MetabolicGeneGraphKernel(model, beta=beta)

    metabolic_df = msd.compute_all_similarities(connected_only=True)

    metabolic_df = metabolic_df.merge(
        metabolic_df,
        left_on=["gene_start", "gene_stop"],
        right_on=["gene_stop", "gene_start"],
        suffixes=["", "_reverse"],
        how="left",
    )

    K_metabolic = metabolic_df.pivot(
        index="gene_start", columns="gene_stop", values="diffusion_similarity"
    )

    K = K_metabolic.values
    k_diag = np.diag(K)
    dist_squared = k_diag[:, None] + k_diag[None, :] - 2 * K
    dist_squared = np.clip(dist_squared, 0, None)
    D = np.sqrt(dist_squared)
    df_dist = pd.DataFrame(D, index=K_metabolic.index, columns=K_metabolic.columns)

    metabolic_df = (
        df_dist.stack()
        .to_frame("distance_metabolic")
        .reset_index()
        .loc[lambda x: x["gene_start"] != x["gene_stop"]]
    )

    joint_dists = metabolic_df.merge(
        transcript_distances,
        left_on=["gene_start", "gene_stop"],
        right_on=["gene1", "gene2"],
    )
    fig = (
        gg.ggplot(joint_dists, gg.aes(x="distance_metabolic", y="distance_pc"))
        + gg.geom_point(size=0.5)
        + gg.theme_minimal()
        + gg.labs(x="Metabolic distance", y="Transcript distance", title=f"beta = {beta}")
        + SHARED_THEME
        + gg.stat_smooth(method="loess", se=True, color="blue")
        + gg.theme(figure_size=(3, 2))
    )
    display(fig)

    corr_ = stats.spearmanr(joint_dists["distance_metabolic"], joint_dists["distance_pc"])[0]

    inter_rows = np.intersect1d(K_metabolic.index, transcript_pairwise_pc.index)
    K_metabolic_ = K_metabolic.loc[inter_rows].loc[:, inter_rows]
    K_transcript_ = transcript_pairwise_pc.loc[inter_rows].loc[:, inter_rows]

    cka_ = compute_cka(K_metabolic_, K_transcript_)
    cosine_similarity_ = compute_cosine_similarity(K_metabolic_, K_transcript_)

    plot_df.append(
        {"beta": beta, "corr": corr_, "cka": cka_, "cosine_similarity": cosine_similarity_}
    )

plot_df = pd.DataFrame(plot_df)

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(x="beta", y="cka"))
    + gg.geom_point()
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(figure_size=(3, 2))
)

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(x="beta", y="cosine_similarity"))
    + gg.geom_point()
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(figure_size=(3, 2))
)

In [ ]:
(
    gg.ggplot(plot_df, gg.aes(x="beta", y="corr"))
    + gg.geom_point()
    + gg.theme_minimal()
    + SHARED_THEME
    + gg.theme(figure_size=(3, 2))
)

### run model for selected $\beta$

In [ ]:
msd = MetabolicGeneGraphKernel(model, beta=25)

metabolic_df = msd.compute_all_similarities(connected_only=True)

print("# of genes", len(msd.genes))
print("# of connected genes", len(msd.connected_genes))

metabolic_df = metabolic_df.merge(
    metabolic_df,
    left_on=["gene_start", "gene_stop"],
    right_on=["gene_stop", "gene_start"],
    suffixes=["", "_reverse"],
    how="left",
)

df_wide = metabolic_df.pivot(index="gene_start", columns="gene_stop", values="diffusion_similarity")

K = df_wide.values
k_diag = np.diag(K)
dist_squared = k_diag[:, None] + k_diag[None, :] - 2 * K
dist_squared = np.clip(dist_squared, 0, None)
D = np.sqrt(dist_squared)
df_dist = pd.DataFrame(D, index=df_wide.index, columns=df_wide.columns)

metabolic_df = (
    df_dist.stack()
    .to_frame("distance_metabolic")
    .reset_index()
    .loc[lambda x: x["gene_start"] != x["gene_stop"]]
)

joint_dists = metabolic_df.merge(
    transcript_distances,
    left_on=["gene_start", "gene_stop"],
    right_on=["gene1", "gene2"],
)

### metabolic distance clustermaps

In [ ]:
vmax = np.quantile(df_dist.values.flatten(), 0.9)
sns.clustermap(df_dist, cmap="rocket_r", vmax=vmax)
plt.savefig("metabolic_distance_clustermap_beta25.png")
plt.show()

### Clustering coprojection

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, metric="precomputed", init="random")
tsne_results = tsne.fit_transform(df_dist)
df_tsne = pd.DataFrame(tsne_results, index=df_dist.index, columns=["TSNE1", "TSNE2"]).assign(
    gene_name=lambda x: x.index
)

In [ ]:
import plotly.express as px


fig = px.scatter(
    df_tsne,
    x="TSNE1",
    y="TSNE2",
    hover_name="gene_name",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
# transcriptomic-derived annotations
gene_transcript_annotation = (
    adata_case.obs.groupby("gene")["annotated_leiden_case"]
    .agg(lambda x: x.mode()[0])
    .to_frame("transcript_annotation")
)
gene_transcript_annotation_coarse = (
    adata_case.obs.groupby("gene")["annotated_leiden_case_coarse"]
    .agg(lambda x: x.mode()[0])
    .to_frame("transcript_annotation_coarse")
)
gene_transcript_annotation = gene_transcript_annotation.merge(
    gene_transcript_annotation_coarse, left_index=True, right_index=True, how="left"
)

df_tsne_ = df_tsne.merge(gene_transcript_annotation, left_index=True, right_index=True, how="left")
df_tsne_["transcript_annotation_coarse"] = (
    df_tsne_["transcript_annotation_coarse"].astype(str).fillna("other")
)

In [ ]:
(
    gg.ggplot(df_tsne_, gg.aes(x="TSNE1", y="TSNE2", color="transcript_annotation_coarse"))
    + gg.geom_point(size=0.5)
    + gg.theme_minimal()
    + gg.labs(x="TSNE1", y="TSNE2", color="transcript_annotation_coarse")
    + gg.theme(figure_size=(3, 2), legend_position="none")
)

In [ ]:
import json

with open("gene_llm_annotations2.json", "r") as f:
    gene_llm_annotations = json.load(f)

gene_llm_annotations_df = pd.Series(gene_llm_annotations).to_frame("llm_annotation")

df_tsne_ = df_tsne.merge(gene_llm_annotations_df, left_index=True, right_index=True, how="left")

In [ ]:
df_tsne_

In [ ]:
fig = (
    gg.ggplot(df_tsne_, gg.aes(x="TSNE1", y="TSNE2", color="llm_annotation"))
    + gg.geom_point(size=0.1)
    + gg.theme_minimal()
    + gg.labs(x="TSNE1", y="TSNE2", color="LLM annotation")
    + SHARED_THEME
    + gg.theme(figure_size=(4, 2), legend_key_size=1)
    + gg.scale_color_cmap_d(cmap_name="tab10")
)
fig

In [ ]:
spearman_corr = stats.spearmanr(joint_dists["distance_metabolic"], joint_dists["distance_pc"])[0]

fig = (
    gg.ggplot(joint_dists, gg.aes(x="distance_metabolic", y="distance_pc"))
    + gg.geom_point(size=0.5)
    + gg.stat_smooth(method="loess", se=True, color="blue")
    + gg.theme_minimal()
    + gg.labs(
        x="Metabolic distance", y="Transcript distance", title=f"Spearman rho = {spearman_corr:.2f}"
    )
    + SHARED_THEME
    + gg.theme(figure_size=(3, 2))
)
# fig.save("03182026_metabolic_transcript_distance.png", dpi=300, bbox_inches="tight")
fig

In [ ]:
# spearman_corr = stats.spearmanr(joint_dists["distance_metabolic"], joint_dists["distance_pc"])[0]

# fig = (
#     gg.ggplot(joint_dists, gg.aes(x="distance_metabolic", y="distance_pc"))
#     + gg.geom_point(size=0.5)
#     + gg.stat_smooth(method="loess", se=True, color="blue")
#     + gg.theme_minimal()
#     + gg.labs(
#         x="Metabolic distance", y="Transcript distance", title=f"Spearman rho = {spearman_corr:.2f}"
#     )
#     + SHARED_THEME
#     + gg.scale_x_log10()
#     + gg.theme(figure_size=(3, 2))
# )
# fig
# # fig.save("03182026_metabolic_transcript_distance.png", dpi=300, bbox_inches="tight")

In [ ]:
joint_dists

In [ ]:
import plotly.express as px


fig = px.scatter(
    joint_dists,
    x="distance_metabolic",
    y="distance_pc_case",
    hover_name="gene_pair_x",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
import plotly.express as px


fig = px.scatter(
    joint_dists,
    x="distance_metabolic",
    y="distance_pc",
    hover_name="gene_pair_x",
    template="plotly_white",
    width=600,
    height=400,
)
fig.update_traces(marker=dict(size=3))
fig.show()

In [ ]:
gene1 = "ilvB"
gene2 = "ilvl"

obs_subset = adata_case.obs.loc[adata_case.obs["gene"].isin([gene1, gene2])]
obs_subset["gene"] = obs_subset["gene"].astype(str)
(
    gg.ggplot(adata_case.obs, gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2"))
    + gg.geom_point(alpha=0.1)
    + gg.geom_point(
        obs_subset,
        gg.aes(x="transcript_case_UMAP1", y="transcript_case_UMAP2", color="gene"),
    )
    + gg.theme_minimal()
    + gg.labs(x="UMAP1", y="UMAP2")
    + gg.theme(figure_size=(3, 2))
)

In [ ]:
(
    joint_dists.loc[lambda x: x.distance_metabolic <= 1e-1]
    .sort_values("distance_pc_case", ascending=False)
    .head(10)
)

In [ ]:
def print_substrates_products(gene):
    substrates = [msd.idx_to_metabolite[x] for x in msd.gene_substrates[gene]]
    products = [msd.idx_to_metabolite[x] for x in msd.gene_products[gene]]
    print(f"{gene} substrates: {', '.join(substrates)}")
    print(f"{gene} products: {', '.join(products)}")

In [ ]:
print_substrates_products("alaS")
print_substrates_products("murC")

In [ ]:
atpC, pstB
compute_similarity

In [ ]:
kxx = msd.compute_similarity("atpC", "atpC")
kyy = msd.compute_similarity("pstB", "pstB")
kxy = msd.compute_similarity("atpC", "pstB")

In [ ]:
dist = np.sqrt(kxx + kyy - 2 * kxy)
print("K(atpC, atpC) =", kxx)
print("K(pstB, pstB) =", kyy)
print("K(atpC, pstB) =", kxy)
print("Distance =", dist)